In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
ollama_api_key = 'ollama'

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:8]}")
else:
    print("DeepSeek API Key not set")

In [ ]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

ollama_url = "http://localhost:11434/v1"
anthropic_url = "https://api.anthropic.com/v1/"
deepseek_url = "https://api.deepseek.com"
# gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
# gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
DONT_ALLUCINATE_PROMPT = "Always be accurate. If you don't know the answer, say so."
MARKDOWN_PROMPT = "Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."

In [ ]:
SYSTEM_PROMPT = " ".join([
    "You are a basic assistant.",
    "You solve simple tasks.",
    "Your answers are brief and straight to the point.",
    DONT_ALLUCINATE_PROMPT,
    MARKDOWN_PROMPT,
])

In [ ]:
MODEL = "gpt-4.1-mini"
# MODEL = "claude-sonnet-4-5-20250929"
# MODEL = "deepseek-chat"
# MODEL = "llama3.2"
# MODEL = "gpt-oss:20b"

In [ ]:
DB = "vehicle_types.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS vehicle_types (vehicle_name TEXT PRIMARY KEY, vehicle_type TEXT)')
    conn.commit()

In [ ]:
vehicle_type_function = {
    "name": "get_vehicle_type_info",
    "description": "Get information about the type of a vehicle.",
    "parameters": {
        "type": "object",
        "properties": {
            "vehicle_name": {"type": "string", "description": "The name of the vehicle the user wants to know the type of"}
        }
    },
    "required": ["vehicle_name"],
    "additional_properties": False,
}

In [ ]:
tools = [{"type": "function", "function": vehicle_type_function}]

In [ ]:
def set_vehicle_types(vehicle_name, vehicle_type):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO vehicle_types (vehicle_name, vehicle_type) VALUES (?, ?) ON CONFLICT(vehicle_name) DO UPDATE SET vehicle_type = ?', (vehicle_name.lower(), vehicle_type.lower(), vehicle_type.lower()))
        conn.commit()

In [ ]:
vehicles_info = {
    "zx-6r": "Sportbike",
    "mt-09": "Naked bike",
    "nx500": "Adventure bike",
    "ktm 390 adventure": "Adventure bike",
}

for vehicle_name, vehicle_type in vehicles_info.items():
    set_vehicle_types(vehicle_name, vehicle_type)

In [ ]:
def get_vehicle_type_info(vehicle_name):
    print(f"DB tool called with {vehicle_name}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT vehicle_type FROM vehicle_types WHERE vehicle_name = ?', (vehicle_name.lower(),))
        result = cursor.fetchone()
        return f"The {vehicle_name} is a {result[0]}" if result else "Unknown vehicle type"


In [ ]:
def handle_tool_calls_and_return_vehicle_names(message):
    responses = []
    vehicle_names = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_vehicle_type_info":
            arguments = json.loads(tool_call.function.arguments)
            vehicle_name = arguments.get('vehicle_name')
            vehicle_names.append(vehicle_name)
            vehicle_type = get_vehicle_type_info(vehicle_name)
            responses.append({
                "role": "tool",
                "content": vehicle_type,
                "tool_call_id": tool_call.id
            })
    return responses, vehicle_names

In [ ]:
# import base64
# import requests

# from io import BytesIO
# from PIL import Image

In [ ]:
# def artist(vehicle_name):
#     result = openai.images.generate(
#         model="gpt-image-1",
#         prompt=f"An image representing a driver on a {vehicle_name}, doing typical activities you'd do on a {vehicle_name}, in an epic pump-up style",
#         size="1024x1024",
#     )

#     image_bytes = base64.b64decode(result.data[0].b64_json)
#     return Image.open(BytesIO(image_bytes))

In [ ]:
# image = artist("ZX-6R")
# display(image)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

In [ ]:
def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    total_tool_calls = 0
    vehicle_names = []
    image = None
    voice = None

    while response.choices[0].finish_reason == "tool_calls" and total_tool_calls < 10:
        message = response.choices[0].message

        responses, new_vehicle_names = handle_tool_calls_and_return_vehicle_names(message)

        vehicle_names.extend(new_vehicle_names)

        messages.append(message)
        messages.extend(responses)

        current_tool_calls = len(message.tool_calls)
        total_tool_calls += current_tool_calls

        print(f"Total tool calls: {total_tool_calls}")

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    reply = response.choices[0].message.content

    history.append({
        "role": "assistant",
        "content": reply
    })

    voice = talker(reply)

    # if vehicle_names:
    #     image = artist(vehicle_names[0])

    return history, voice, image

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        # image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        # chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
        chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )

ui.launch(inbrowser=True)